# 6-DOF UAV Simulator: Walkthrough

An end-to-end tour of the simulator: aircraft model, trim, linearization,
autopilot tracking, waypoint guidance with L1 loiter, Dryden turbulence
with a Monte Carlo sweep, and a stability sweep across airspeeds.

Install the package once with `pip install -e ".[dev]"` from the repo root,
then run cells top to bottom.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from uavsim import (
    Autopilot, CommandSchedule, DrydenGust, DrydenWind,
    WaypointCommands, classify_modes, default_uav,
    flight_eigenvalues, monte_carlo, simulate,
    simulate_closed_loop, stability_sweep, trim,
)
from uavsim.viz import plot_controls, plot_state_history

## 1. Aircraft model

Parameters live on a single `Aircraft` dataclass: mass, geometry, inertia,
Cessna-172-class stability derivatives (Nelson, Appendix B), and propulsion.
Sign convention: +δe pitches nose down, +δa rolls right, +δr yaws nose left,
δt ∈ [0, 1] scales max thrust.

In [ ]:
ac = default_uav()
print(f'mass = {ac.mass} kg, S = {ac.S} m², b = {ac.b} m, c = {ac.c} m')
print(f'I = diag({ac.Ix}, {ac.Iy}, {ac.Iz}) kg·m², T_max = {ac.T_max} N')

## 2. Trim

Find a steady, wings-level, constant-altitude state at a chosen airspeed
and altitude. The solver works in (α, δe, δt) and zeros the body-frame
x and z forces and the pitching moment.

In [ ]:
ts = trim(ac, Va=ac.U0, alt=ac.alt0)
print(f'Va  = {ts.Va:.2f} m/s,   alt = {ts.alt:.1f} m')
print(f'α   = {np.rad2deg(ts.alpha):+.3f}°')
print(f'θ   = {np.rad2deg(ts.theta):+.3f}°')
print(f'δe  = {np.rad2deg(ts.u[0]):+.3f}°')
print(f'δt  = {ts.u[3]:.3f}')

## 3. Linearization and flight modes

Numerically perturb the 6-DOF EOM at trim to compute A = ∂f/∂x. The
non-trivial eigenvalues are the classical longitudinal and lateral modes.
Position and yaw show up as zeros (integrators) and are dropped.

In [ ]:
eigs = flight_eigenvalues(ac, ts)
modes = classify_modes(eigs)
print(f"{'mode':14s}  {'eigenvalue':>22s}   ω_n [rad/s]   ζ")
for m in modes:
    if m.is_oscillatory:
        print(f'  {m.name:12s}  {m.eig.real:+8.4f}{m.eig.imag:+8.4f}j  '
              f'{m.omega_n:8.3f}     {m.zeta:+.3f}')
    else:
        print(f'  {m.name:12s}  {m.eig.real:+8.4f}             '
              f'{m.omega_n:8.3f}    (real)')

## 4. Open-loop trimmed flight

If trim is correct, propagating the trimmed state with the trimmed controls
should hold the state forever (within integrator tolerance).

In [ ]:
res = simulate(ac, ts.x, t_end=30.0, control=ts.u, dt_log=0.05)
print(f'Va  drift in 30 s: {abs(res.Va[-1] - ts.Va):.4f} m/s')
print(f'alt drift in 30 s: {abs(res.altitude[-1] - ts.alt):.4f} m')
fig = plot_state_history(res, title='Open-loop trimmed flight')
fig.set_size_inches(10, 7); plt.show()

## 5. Closed-loop autopilot

The 7-loop cascaded autopilot tracks altitude, heading, and airspeed
commands. Inner loops handle pitch, roll, sideslip; a yaw damper provides
dutch-roll damping.

In [ ]:
h0, psi0, Va0 = ts.alt, 0.0, ts.Va
cmd = CommandSchedule.from_dict({
    'altitude': lambda t: h0 + 50.0 * (t >= 5.0),
    'heading':  lambda t: psi0 + np.deg2rad(30.0) * (t >= 60.0),
    'airspeed': lambda t: Va0 + 2.0 * (t >= 100.0),
})
ap = Autopilot(ac, ts, cmd)
log = simulate_closed_loop(ac, ts, ap, t_end=200.0, dt_ctrl=0.02)
r = log.res
print(f'final tracking errors:')
print(f'  altitude = {abs(r.altitude[-1] - (h0+50)):.3f} m')
print(f'  heading  = {abs(np.rad2deg(r.euler[-1,2]) - 30):.3f}°')
print(f'  Va       = {abs(r.Va[-1] - (Va0+2)):.3f} m/s')

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
t = r.t
ax[0].plot(t, [cmd.altitude(ti) for ti in t], 'k--', lw=1, label='cmd')
ax[0].plot(t, r.altitude, label='meas'); ax[0].set_ylabel('altitude [m]'); ax[0].legend()
ax[1].plot(t, np.rad2deg([cmd.heading(ti) for ti in t]), 'k--', lw=1)
ax[1].plot(t, np.rad2deg(r.euler[:,2])); ax[1].set_ylabel('heading [deg]')
ax[2].plot(t, [cmd.airspeed(ti) for ti in t], 'k--', lw=1)
ax[2].plot(t, r.Va); ax[2].set_ylabel('Va [m/s]'); ax[2].set_xlabel('t [s]')
fig.tight_layout(); plt.show()

## 6. Waypoint mission with L1 loiter

Visit three waypoints, then loiter at the third on an 80 m circle. The
L1 lookahead distance is set by default to 1.25 × loiter_radius, which
makes the orbit collapse to the target radius almost exactly.

In [ ]:
wps = np.array([
    [300.0, 200.0, 110.0],
    [300.0, 600.0, 120.0],
    [600.0, 600.0, 110.0],
])
wp_cmd = WaypointCommands(waypoints=wps, airspeed=ts.Va,
                          arrival_radius=40.0, loiter_radius=80.0)
ap = Autopilot(ac, ts, wp_cmd)
log = simulate_closed_loop(ac, ts, ap, t_end=300.0, dt_ctrl=0.02)
r = log.res
mask = r.t > 200
radii = np.hypot(r.pos_ned[mask, 0] - wps[-1,0], r.pos_ned[mask, 1] - wps[-1,1])
print(f'L1 = {wp_cmd.L1_distance:.1f} m')
print(f'loiter radius (steady state): mean = {radii.mean():.2f}  std = {radii.std():.4f}  target = 80')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(r.pos_ned[:,1], r.pos_ned[:,0], lw=1.2)
ax.scatter(wps[:,1], wps[:,0], c='r', s=50, zorder=3, label='waypoints')
for i, w in enumerate(wps): ax.annotate(f'  WP{i+1}', (w[1], w[0]))
th = np.linspace(0, 2*np.pi, 200)
ax.plot(wps[-1,1] + 80*np.cos(th), wps[-1,0] + 80*np.sin(th), 'k--', alpha=0.4, label='target loiter')
ax.set_xlabel('East [m]'); ax.set_ylabel('North [m]')
ax.set_aspect('equal'); ax.grid(alpha=0.4); ax.legend(); plt.show()

## 7. Dryden turbulence and Monte Carlo sensitivity

MIL-F-8785C 2nd-order shaping for v and w; 1st-order Markov for u.
Monte Carlo sweeps gust intensity and reports tracking-error statistics.

In [ ]:
def build_ap():
    cmd = CommandSchedule.from_dict({
        'altitude': lambda t: ts.alt,
        'heading':  lambda t: 0.0,
        'airspeed': lambda t: ts.Va,
    })
    return Autopilot(ac, ts, cmd)

intensities = [0.25, 0.5, 1.0, 1.5, 2.0]
n_runs = 10
results = monte_carlo(ac, ts, build_ap, intensities=intensities,
                      n_runs=n_runs, t_end=20.0, dt_ctrl=0.02)
for s in results:
    z = s.summary()
    print(f"σ scale = {z['intensity']:.2f}: "
          f"alt RMS = {z['alt_rms_mean']:.3f} m, "
          f"hdg RMS = {np.rad2deg(z['hdg_rms_mean']):.3f}°, "
          f"Va RMS = {z['Va_rms_mean']:.3f} m/s")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
labels = [str(i) for i in intensities]
axes[0].boxplot([r.altitude_rms for r in results], tick_labels=labels)
axes[0].set_xlabel('gust intensity'); axes[0].set_ylabel('alt RMS [m]')
axes[1].boxplot([np.rad2deg(r.heading_rms) for r in results], tick_labels=labels)
axes[1].set_xlabel('gust intensity'); axes[1].set_ylabel('hdg RMS [deg]')
axes[2].boxplot([r.Va_rms for r in results], tick_labels=labels)
axes[2].set_xlabel('gust intensity'); axes[2].set_ylabel('Va RMS [m/s]')
fig.tight_layout(); plt.show()

## 8. Stability sweep across airspeeds

Trim and linearize at each airspeed in a grid and plot the eigenvalue
locus. The phugoid migrates left to right as V increases; the dutch roll
and short period also shift.

In [ ]:
V_grid = np.linspace(13.0, 30.0, 18)
out = stability_sweep(ac, V_grid)

fig, ax = plt.subplots(figsize=(8, 7))
cmap = plt.cm.viridis
for V, eigs in zip(out['V_grid'], out['eigs']):
    c = cmap((V - V_grid.min()) / (V_grid.max() - V_grid.min()))
    ax.scatter(eigs.real, eigs.imag, c=[c], s=18, alpha=0.85)
ax.axvline(0, color='r', ls='--', alpha=0.4)
ax.set_xlabel('Re(λ) [1/s]'); ax.set_ylabel('Im(λ) [rad/s]')
ax.set_title('Eigenvalue locus vs. airspeed (color = V)')
ax.grid(alpha=0.4); plt.show()

## Closing

From here you can:

* Tune autopilot gains in `src/uavsim/autopilot.py:APGains`.
* Swap the airframe by editing `default_uav()` in `src/uavsim/aircraft.py`.
* Run the interactive 3D animation: `python examples/animate_mission.py`.
* Run the full end-to-end demo: `python examples/demo.py`.